# hypertopos on Spark: quickstart

From a table of payments to the accounts that stand out from their population, why they do, and how to keep scoring new data against the same baseline. Runs on Databricks (Runtime 16+ or serverless environment 3+) or on a local Spark session. The data is synthetic and labelled, so you can see which flagged accounts take part in a laundering pattern.

On Databricks PySpark is already installed, so install hypertopos alone. hypertopos needs Python 3.12: on serverless, pick environment version 3 or later in the **Environment** side panel first. Locally, install it with the Spark extra instead: `%pip install "hypertopos[spark]"`.

In [ ]:
%pip install hypertopos

In [ ]:
# On Databricks `spark` already exists; elsewhere, start a session.
if "spark" not in globals():
    from pyspark.sql import SparkSession

    spark = SparkSession.builder.getOrCreate()

## 1. Data

`synthetic_aml` generates retail and business accounts and their payments over 60 days. About 1 % of the accounts take part in one of six laundering patterns (fan-out, fan-in, cycle, scatter-gather, structuring, pass-through). Each payment carries the sender's segment, so the accounts can be compared within their segment.

In [ ]:
from hypertopos.datasets import synthetic_aml

data = synthetic_aml(n_accounts=5_000, seed=1)
tx = spark.createDataFrame(data.transactions.to_pandas())
accounts = spark.createDataFrame(data.accounts.to_pandas())
events = tx.join(accounts.select("account", "segment"), tx.from_account == accounts.account)
events.select("ts", "from_account", "to_account", "amount", "segment").show(5)

## 2. Find anomalies

Four column roles: who the event belongs to, the other side, the amount and the time. `group_by="segment"` compares business accounts with businesses and retail with retail.

In [ ]:
from hypertopos.spark import find_anomalies

result = find_anomalies(
    events,
    entity="from_account",
    counterparty="to_account",
    amount="amount",
    time="ts",
    group_by="segment",
)
result.orderBy("delta_norm", ascending=False).show(10, truncate=False)

`delta_norm` is the distance from the segment's centre in standard deviations, `is_anomaly` marks the accounts beyond the segment's 99th percentile, `p_value` is the share of the population at least as far out, and `top_reasons` names the dimensions that move the account most.

## 3. Compare with the labels

The threshold flags about 1 % of every population by construction, so the flagged set always has a size; the labels show what it contains.

In [ ]:
from pyspark.sql import functions as F

labels = accounts.select(F.col("account").alias("entity"), "is_laundering_account", "typology")
flagged = result.where("is_anomaly").join(labels, "entity", "left")
flagged.groupBy("segment", "is_laundering_account").count().orderBy("segment").show()
flagged.where("is_laundering_account").groupBy("typology").count().show()

## 4. Why an account stands out

`explain` gives one row per account and dimension: the value, the segment's mean and standard deviation, the signed deviation and its share of the distance.

In [ ]:
from hypertopos.spark import explain

explain(result, top_n=3).show(15, truncate=False)

## 5. Score the next period against the same baseline

`score` measures new events against the population `result` was calibrated on, without recalibrating, so scores stay comparable from one period to the next. Here a second synthetic period stands in for next month's payments.

In [ ]:
from hypertopos.spark import score

later = synthetic_aml(n_accounts=5_000, seed=2)
later_tx = spark.createDataFrame(later.transactions.to_pandas())
later_accounts = spark.createDataFrame(later.accounts.to_pandas())
later_events = later_tx.join(
    later_accounts.select("account", "segment"), later_tx.from_account == later_accounts.account
)
scored = score(later_events, like=result)
scored.where("is_anomaly").orderBy("delta_norm", ascending=False).show(10, truncate=False)

## 6. Hand the result to an agent

`sphere_of` returns the calibration behind the result: its geometry as a table, or a sphere for the [hypertopos MCP server](https://github.com/hypertopos/hypertopos-mcp) with the same distances and flags, so an agent can investigate the flagged accounts.

In [ ]:
from hypertopos.spark import sphere_of

sphere = sphere_of(result)
geometry = sphere.geometry()  # pyarrow table: values, deviations and scores per account
print(geometry.num_rows, "accounts,", len(geometry.column_names), "columns")
sphere.mcp_config()  # put under mcpServers in an MCP client's configuration